# Lab 05

### **Uwaga**
W poniższych zadaniach zakładamy, iż serwer powinien obsługiwać tylko jednego klienta w danej chwili.

In [ ]:
import socket
import time
import statistics

HOST = "127.0.0.1"

1. Pod adresem 212.182.24.27 na porcie TCP o numerze 2912 działa serwer losujący liczby. Napisz program klienta, który będzie pobierał od użytkownika liczbę, a następnie będzie wysyłał ją do serwera w celu odgadnięcia wylosowanej przez serwer liczby. Po wysłaniu liczby klient powinien odbierać od serwera odpowiedź mówiącą o tym, czy udało nam się daną liczbę odgadnąć.

In [ ]:
PORT_EX1 = 2901

In [ ]:
def ex01():
    print("=== TCP Client - Number Guessing Game ===")
    print(f"Connecting to {HOST}:{PORT_EX1}...\n")

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.connect((HOST, PORT_EX1))
        print("Connected to server!\n")

        try:
            s.settimeout(2.0)
            welcome = s.recv(1024)
            if welcome:
                print(f"Server: {welcome.decode('utf-8', errors='replace').strip()}")
        except socket.timeout:
            pass
        finally:
            s.settimeout(None)

        while True:
            number = input("\nEnter a number to guess (or 'q' to quit): ").strip()

            if number.lower() == 'q':
                print("Client terminated.")
                break

            s.sendall((number + "\n").encode('utf-8'))

            response = s.recv(1024).decode('utf-8', errors='replace').strip()
            print(f"Server response: {response}")

            if "guessed" in response.lower() or "correct" in response.lower():
                print("\nCongratulations! You guessed the number!")
                break

2. Napisz program serwera, który działając pod adresem 127.0.0.1 oraz na określonym porcie TCP będzie
losował liczbę i odbierał od klienta wiadomości. W przypadku, gdy w wiadomości klient przyśle do serwera
coś innego, niż liczbę, serwer powinien poinformować klienta o błędzie. Po odebraniu liczby od klienta
serwer sprawdza, czy otrzymana liczba jest:
- mniejsza od wylosowanej przez serwer
- równa wylosowanej przez serwer
- większa od wylosowanej przez serwer

A następnie odsyła stosowną informację do klienta. W przypadku, gdy klient odgadnie liczbę, serwer
powinien zakończyć działanie.

In [ ]:
PORT_EX2 = 2902

In [ ]:
def ex02():
    print("=== TCP Client - Number Guessing Game ===")
    print(f"Connecting to {HOST}:{PORT_EX2}...\n")

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.connect((HOST, PORT_EX2))
        print("Connected!\n")

        try:
            s.settimeout(2.0)
            welcome = s.recv(1024)
            if welcome:
                print(f"Server: {welcome.decode('utf-8', errors='replace').strip()}")
        except socket.timeout:
            pass
        finally:
            s.settimeout(None)

        while True:
            number = input("\nEnter a number to guess (or 'q' to quit): ").strip()

            if number.lower() == "q":
                print("Client terminated.")
                break

            s.sendall((number + "\n").encode("utf-8"))

            response = s.recv(1024).decode("utf-8", errors="replace").strip()
            print(f"Server: {response}")

            if "correct" in response.lower():
                print("Congratulations! You guessed the number!")
                break

3. ***Port-knocking*** jest metodą pozwalającą na nawiązanie zdalnego połączenia z usługami działającymi
na komputerze, do którego dostęp został ograniczony np. za pomocą zapory sieciowej, umożliwiającą
odróżniania prób połączeń, które powinny i nie powinny być zrealizowane. Inaczej mówiąc, to metoda
ustanawiania połączenia z hostem o zamkniętych portach.

Pod adresem 212.182.24.27 na porcie TCP o numerze 2913 działa ukryta usługa. Usługa jest zabezpieczona
metodą port knocking -> po otrzymaniu od klienta odpowiedniej sekwencji pakietów UDP na odpowiednie
porty, otwiera wspomniany wyżej port TCP. Napisz program klienta, który odgadnie sekwencję portów
UDP, a następnie odbierze od serwera wiadomość na porcie TCP.

**Uwaga:** Aby znaleźć porty UDP, składające się na sekwencję otwarcia docelowego portu TCP, wysyłaj
do serwera wiadomość o treści PING. W przypadku, gdy uda się znaleźć port UDP, należący do sekwencji
otwierającej port TCP, serwer odeśle wiadomość PONG. Porty UDP, które wchodzą w skład sekwencji,
kończą się na 666. Usługa działająca na ukrytym porcie, jeśli uda się ją znaleźć, zwraca w odpowiedzi
tekst: ```Congratulations! You found the hidden.```

In [ ]:
TCP_PORT_EX3 = 2903
UDP_MESSAGE = b"PING"
EXPECTED_RESPONSE = "PONG"
UDP_TIMEOUT = 1.0

In [ ]:
def get_ports_ending_with_666():
    ports = []
    n = 666
    while n <= 65535:
        ports.append(n)
        n += 1000
    return ports


def send_ping(port):
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
        s.settimeout(UDP_TIMEOUT)
        try:
            s.sendto(UDP_MESSAGE, (HOST, port))
            data, _ = s.recvfrom(1024)
            return data.decode("utf-8", errors="replace").strip()
        except (socket.timeout, OSError):
            return None


def connect_tcp():
    print(f"\n[TCP] Trying to connect to {HOST}:{TCP_PORT_EX3}...")
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(5.0)
            s.connect((HOST, TCP_PORT_EX3))
            print("[TCP] Connected!")
            message = s.recv(4096).decode("utf-8", errors="replace").strip()
            return message
    except Exception as e:
        return f"TCP error: {e}"


def ex03():
    print("=== Port-Knocking Client ===")
    print(f"Server: {HOST}, Target TCP port: {TCP_PORT_EX3}")
    print("Scanning UDP ports ending with 666...\n")

    ports = get_ports_ending_with_666()
    print(f"Ports to check: {len(ports)}")
    print(f"First few: {ports[:5]}...\n")

    sequence = []

    for port in ports:
        print(f"Checking UDP port {port}...", end=" ", flush=True)
        response = send_ping(port)

        if response == EXPECTED_RESPONSE:
            print(f"PONG! Port {port} is part of the sequence!")
            sequence.append(port)

            message = connect_tcp()
            if "Congratulations" in message or "hidden" in message.lower():
                print(f"\n[SUCCESS] Message from TCP server: {message}")
                print(f"Knock sequence: {sequence}")
                return
            else:
                print(f"[TCP] Response (sequence incomplete?): {message}")
        else:
            print("no response")

        time.sleep(0.05)

    if sequence:
        print(f"\nFound UDP ports (PONG): {sequence}")
        print("Final TCP connection attempt...")
        message = connect_tcp()
        print(f"TCP response: {message}")
    else:
        print("\nNo ports belonging to the sequence were found.")

4. Napisz parę programów -> klienta i serwer, w których porównasz czas przesyłu pakietów za pomocą gniazda
TCP i gniazda UDP. Następnie, po przeprowadzonym teście, odpowiedz na pytania:
- Dla którego z gniazd czas jest krótszy?
- Z czego wynika krótszy czas?
- Jakie są zalety / wady obu rozwiązań?

In [ ]:
TCP_PORT_EX4 = 2904
UDP_PORT_EX4 = 2905
PACKET_COUNT = 1000
PACKET_SIZE = 1024
UDP_TIMEOUT = 2.0

In [ ]:
def test_tcp(count, size):
    times = []
    message = b"X" * size

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.connect((HOST, TCP_PORT_EX4))

        for _ in range(count):
            start = time.perf_counter()
            s.sendall(message)

            received = 0
            while received < size:
                data = s.recv(size - received)
                if not data:
                    raise ConnectionError("Server closed connection")
                received += len(data)

            times.append((time.perf_counter() - start) * 1000)

    return times


def test_udp(count, size):
    times = []
    lost = 0
    message = b"X" * size

    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
        s.settimeout(UDP_TIMEOUT)

        for _ in range(count):
            start = time.perf_counter()
            try:
                s.sendto(message, (HOST, UDP_PORT_EX4))
                s.recvfrom(65535)
                times.append((time.perf_counter() - start) * 1000)
            except socket.timeout:
                lost += 1

    return times, lost


def print_stats(label, times, lost=0):
    print(f"\n{'=' * 50}")
    print(f"  {label}")
    print(f"{'=' * 50}")
    print(f"  Successful measurements : {len(times)}")
    if lost:
        print(f"  Lost packets            : {lost}")
    if times:
        print(f"  Average RTT             : {statistics.mean(times):.4f} ms")
        print(f"  Median RTT              : {statistics.median(times):.4f} ms")
        print(f"  Min RTT                 : {min(times):.4f} ms")
        print(f"  Max RTT                 : {max(times):.4f} ms")
        if len(times) > 1:
            print(f"  Std deviation           : {statistics.stdev(times):.4f} ms")
        print(f"  Total time              : {sum(times):.2f} ms")
    print(f"{'=' * 50}")


def ex04():
    print("=" * 60)
    print("   TRANSFER TIME BENCHMARK: TCP vs UDP")
    print("=" * 60)
    print(f"Packets     : {PACKET_COUNT}")
    print(f"Packet size : {PACKET_SIZE} bytes")
    print(f"Server      : {HOST}")

    print(f"\n[1/2] Testing TCP (port {TCP_PORT_EX4})...")
    try:
        tcp_times = test_tcp(PACKET_COUNT, PACKET_SIZE)
        print_stats("TCP RESULTS", tcp_times)
    except Exception as e:
        print(f"TCP error: {e}")
        tcp_times = []

    print(f"\n[2/2] Testing UDP (port {UDP_PORT_EX4})...")
    try:
        udp_times, udp_lost = test_udp(PACKET_COUNT, PACKET_SIZE)
        print_stats("UDP RESULTS", udp_times, udp_lost)
    except Exception as e:
        print(f"UDP error: {e}")
        udp_times, udp_lost = [], 0

    if tcp_times and udp_times:
        avg_tcp = statistics.mean(tcp_times)
        avg_udp = statistics.mean(udp_times)

        print("\n" + "=" * 60)
        print("   COMPARISON")
        print("=" * 60)
        print(f"  Average RTT TCP : {avg_tcp:.4f} ms")
        print(f"  Average RTT UDP : {avg_udp:.4f} ms")

        if avg_udp < avg_tcp:
            diff = avg_tcp - avg_udp
            print(f"\n  UDP is faster by {diff:.4f} ms ({diff / avg_tcp * 100:.1f}%)")
        else:
            diff = avg_udp - avg_tcp
            print(f"\n  TCP is faster by {diff:.4f} ms ({diff / avg_udp * 100:.1f}%)")

    print("""
+----------------------------------------------------------+
|                   ANALYSIS & ANSWERS                     |
+----------------------------------------------------------+
|                                                          |
|  1. WHICH SOCKET HAS SHORTER TRANSFER TIME?             |
|     UDP is typically faster, especially on loopback.    |
|                                                          |
|  2. WHY IS UDP FASTER?                                  |
|     - No connection setup (no 3-way handshake)          |
|     - No acknowledgements (ACK)                         |
|     - No flow or congestion control                     |
|     - No retransmission of lost packets                 |
|     - Smaller header (8 B UDP vs 20+ B TCP)             |
|                                                          |
|  3. ADVANTAGES / DISADVANTAGES:                         |
|                                                          |
|  TCP:                                                    |
|    + Guaranteed delivery of data                        |
|    + Preserves packet order                             |
|    + Error control and retransmission                   |
|    + Flow control (prevents receiver overflow)          |
|    - Higher overhead (handshake, ACK, headers)          |
|    - Higher latency                                     |
|    - Head-of-line blocking                              |
|                                                          |
|  UDP:                                                    |
|    + Low latency, higher throughput                     |
|    + Simple protocol                                    |
|    + Supports multicast and broadcast                   |
|    + Suitable for real-time apps (VoIP, games, video)  |
|    - No delivery guarantee                              |
|    - Packets may be lost or reordered                   |
|    - No flow control                                    |
+----------------------------------------------------------+
""")

Do wygodniejszego włączania klientów

In [ ]:
ex01()

In [ ]:
ex02()

In [ ]:
ex03()

In [ ]:
ex04()